# Final scoreboard: every task, against the best honestly evaluated published result

Pre-registered in `docs/PREREG_2026-11-16.md`. No training: this notebook reads saved out-of-fold probabilities
and does every test the plan fixed in advance.

- **B.** The four coupling-graph twins (Pearson, PLV, AEC-c, Granger) against today's GAT. Primary: subject macro
  recall. Key secondary: the AD-vs-FTD AUC inside the 3-class model. Holm over the four twins.
- **C.** The dedicated dementia-vs-HC model, alone and soft-voted, against the 3-class model's by-product.
- **The final 3-class model** is the two-stage model: P(dementia) from C's soft vote and P(AD | dementia) from C3
  (Experiment 10, or Experiment 11's two-recording average if its rule adopted it). It is compared with today's soft
  vote.
- **The scoreboard.** "Level" means the published value lies inside our 95 % CI; "ahead" or "behind" means outside.

Every comparison is paired: the same 88 subjects, the same folds (repeats 1–3 of the 3-class notebook), and the
same 2000 subject bootstraps, with each class resampled separately.

In [1]:
# ===========================================================================
# CELL 1 — INPUTS
# ===========================================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd

WORK_DIR = Path.cwd()
EXP_DIR  = WORK_DIR / "experiments_ad_vs_ftd" / "results"
OUT_DIR  = WORK_DIR / "results_final"
OUT_DIR.mkdir(parents=True, exist_ok=True)
HC, AD, FTD = 0, 1, 2
SEED, N_BOOT, R = 42, 2000, 3            # 3 repeats: the folds every model below shares
GRAPHS = ["pearson", "plv", "aec", "granger"]

_n   = np.array([len(np.load(WORK_DIR / "subject_cache" / f"sub-{i:03d}.npz")["y"]) for i in range(1, 89)])
SIDX = np.repeat(np.arange(88), _n)                                   # window -> subject
Y    = np.array([AD if i <= 36 else HC if i <= 65 else FTD for i in range(1, 89)])


def subj(path):
    """(repeats, windows, K) out-of-fold probabilities -> (3, 88, K): mean over each subject's windows."""
    o = np.load(path)[:R]
    assert o.shape[1] == len(SIDX) and np.isfinite(o).all(), path
    return np.stack([np.stack([np.bincount(SIDX, o[r][:, k]) / _n for k in range(o.shape[2])], -1) for r in range(R)])


M3 = {"GAT, PSD graphs (today)": subj(WORK_DIR / "results_gat_multiclass" / "oof_prob.npy"),
      "soft vote, PSD graphs (today)": subj(WORK_DIR / "results_gat_multiclass" / "ensemble_oof_prob.npy")}
for g in GRAPHS:
    M3[f"GAT, {g} graphs"] = subj(WORK_DIR / f"results_gat_multiclass_{g}" / "oof_prob.npy")
    M3[f"soft vote, {g} graphs"] = subj(WORK_DIR / f"results_gat_multiclass_{g}" / "ensemble_oof_prob.npy")

_dem = {n: subj(WORK_DIR / "results_gat_dementia" / f) for n, f in
        [("dementia GAT", "oof_prob.npy"), ("band power", "floor_oof_prob.npy"), ("dementia soft vote", "ensemble_oof_prob.npy")]}

# stage 2: C3 from Experiment 10, or Experiment 11's average if its pre-registered rule adopted it
_e10 = np.load(EXP_DIR / "c3_ds006036" / "c3_oof.npz", allow_pickle=True)
_e11 = json.loads((EXP_DIR / "c3_both" / "summary.json").read_text(encoding="utf-8"))
if _e11["primary"]["adopted_as_stage_2"]:
    _z = np.load(EXP_DIR / "c3_both" / "c3_both_oof.npz", allow_pickle=True)
    STAGE2 = "C3, both recordings averaged (Experiment 11)"
    Q = _z["q_ad"][list(_z["runs"]).index("both recordings, averaged (primary)")][:R]
else:
    STAGE2 = "C3, ds006036 (Experiment 10)"
    Q = _e10["q_ad"][list(_e10["runs"]).index("ds006036, 44 s (primary)")][:R]
assert (_e10["labels"] == Y).all()
print(f"{len(M3)} 3-class models, {len(_dem)} dementia models; stage 2 = {STAGE2}")


10 3-class models, 3 dementia models; stage 2 = C3, ds006036 (Experiment 10)


In [2]:
# ===========================================================================
# CELL 2 — METRICS AND THE SHARED BOOTSTRAP
# ===========================================================================
_rng  = np.random.default_rng(SEED)
_cls  = {c: np.flatnonzero(Y == c) for c in (HC, AD, FTD)}
BOOT  = np.stack([np.concatenate([_rng.choice(i, len(i)) for i in _cls.values()]) for _ in range(N_BOOT)])
FULL  = np.concatenate(list(_cls.values()))                            # the real sample, same order


def _auc(pos, neg):
    d = pos[:, None] - neg[None, :]
    return (d > 0).mean() + 0.5 * (d == 0).mean()


def three(P, idx):
    """P (3, 88, 3); idx the subjects (with repeats) to score. Metrics averaged over repeats."""
    y, out = Y[idx], []
    for p in P[:, idx]:
        pred = p.argmax(1)
        cm = np.bincount(y * 3 + pred, minlength=9).reshape(3, 3)
        rec = cm.diagonal() / cm.sum(1)
        prec = np.divide(cm.diagonal(), cm.sum(0), out=np.zeros(3), where=cm.sum(0) > 0)
        f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
        s = p[:, AD] / (p[:, AD] + p[:, FTD])
        out.append(dict(acc=(pred == y).mean(), macro_f1=f1.mean(), macro_recall=rec.mean(),
                        recall_HC=rec[HC], recall_AD=rec[AD], recall_FTD=rec[FTD],
                        auc_AD_FTD=_auc(s[y == AD], s[y == FTD]),
                        dementia_auc=_auc(1 - p[y != HC, HC], 1 - p[y == HC, HC])))
    return pd.DataFrame(out).mean()


def dementia(pd_, call, idx):
    """pd_ (3, 88) P(dementia), call (3, 88) bool. Metrics averaged over repeats."""
    y, out = Y[idx], []
    for p, c in zip(pd_[:, idx], call[:, idx]):
        sens, spec = c[y != HC].mean(), (~c[y == HC]).mean()
        out.append(dict(acc=(c == (y != HC)).mean(), bal_acc=(sens + spec) / 2, sensitivity=sens,
                        specificity=spec, auc=_auc(p[y != HC], p[y == HC]),
                        auc_AD_HC=_auc(p[y == AD], p[y == HC]), auc_FTD_HC=_auc(p[y == FTD], p[y == HC])))
    return pd.DataFrame(out).mean()


def evaluate(fn, *args):
    """Real values and the bootstrap distribution (N_BOOT x metrics) of the repeat mean."""
    return fn(*args, FULL), pd.DataFrame([fn(*args, b) for b in BOOT])


def ci(dist, k):
    return np.percentile(dist[k], [2.5, 97.5])


def paired(da, db, k):
    d = da[k] - db[k]
    return dict(diff=None, lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                p=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); out = np.empty_like(p); run = 0.0
    for k, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - k) * p[i])); out[i] = run
    return out


fmt = lambda v, d, k: f"{v[k]:.3f} [{ci(d, k)[0]:.3f}, {ci(d, k)[1]:.3f}]"
print(f"{N_BOOT} paired subject bootstraps, classes resampled separately (HC 29, AD 36, FTD 23)")


2000 paired subject bootstraps, classes resampled separately (HC 29, AD 36, FTD 23)


In [3]:
# ===========================================================================
# CELL 3 — B: COUPLING-GRAPH TWINS VS TODAY'S GAT
# ===========================================================================
EV3 = {n: evaluate(three, P) for n, P in M3.items()}
KEYS3 = ["macro_recall", "acc", "macro_f1", "recall_HC", "recall_AD", "recall_FTD", "auc_AD_FTD", "dementia_auc"]
B_TABLE = pd.DataFrame({n: {k: fmt(v, d, k) for k in KEYS3} for n, (v, d) in EV3.items()}).T
pd.set_option("display.width", 250, "display.max_colwidth", 30)
print("3-class, subject level, mean of repeats 1-3 [95 % CI]\n")
print(B_TABLE[["macro_recall", "acc", "macro_f1", "recall_FTD", "auc_AD_FTD", "dementia_auc"]].to_string())

_ref = EV3["GAT, PSD graphs (today)"]
tests = []
for k in ("macro_recall", "auc_AD_FTD"):                      # primary, key secondary
    rows = [dict(metric=k, twin=f"GAT, {g} graphs", diff=EV3[f"GAT, {g} graphs"][0][k] - _ref[0][k],
                 **{x: v for x, v in paired(EV3[f"GAT, {g} graphs"][1], _ref[1], k).items() if x != "diff"})
            for g in GRAPHS]
    for r, ph in zip(rows, holm([r["p"] for r in rows])):
        r["p_holm"] = ph
    tests += rows
B_TESTS = pd.DataFrame(tests)
print("\neach twin minus today's GAT (paired bootstrap; Holm over the four twins, per metric)\n")
print(B_TESTS.round(3).to_string(index=False))


3-class, subject level, mean of repeats 1-3 [95 % CI]

                                       macro_recall                   acc              macro_f1            recall_FTD            auc_AD_FTD          dementia_auc
GAT, PSD graphs (today)        0.583 [0.509, 0.656]  0.621 [0.545, 0.693]  0.565 [0.487, 0.642]  0.217 [0.101, 0.348]  0.595 [0.472, 0.714]  0.871 [0.791, 0.940]
soft vote, PSD graphs (today)  0.600 [0.512, 0.684]  0.625 [0.538, 0.705]  0.592 [0.502, 0.680]  0.333 [0.174, 0.493]  0.601 [0.464, 0.735]  0.889 [0.818, 0.950]
GAT, pearson graphs            0.550 [0.474, 0.625]  0.572 [0.492, 0.652]  0.537 [0.456, 0.610]  0.275 [0.159, 0.392]  0.562 [0.437, 0.686]  0.861 [0.774, 0.933]
soft vote, pearson graphs      0.601 [0.511, 0.687]  0.621 [0.530, 0.705]  0.594 [0.500, 0.680]  0.348 [0.188, 0.522]  0.589 [0.453, 0.721]  0.898 [0.828, 0.954]
GAT, plv graphs                0.562 [0.490, 0.635]  0.591 [0.515, 0.663]  0.541 [0.462, 0.617]  0.232 [0.130, 0.362]  0.541 [0.426, 0.

In [4]:
# ===========================================================================
# CELL 4 — C: THE DEDICATED DEMENTIA-VS-HC MODEL
# ===========================================================================
DEM = {n: (P[..., 1], P.argmax(-1) == 1) for n, P in _dem.items()}
for n in ("GAT, PSD graphs (today)", "soft vote, PSD graphs (today)"):
    P = M3[n]                                                 # the 3-class model's by-product, as its notebook scores it
    DEM[f"3-class {n.split(',')[0]} by-product"] = (1 - P[..., HC], P.argmax(-1) != HC)
EVD = {n: evaluate(dementia, p, c) for n, (p, c) in DEM.items()}
KEYSD = ["acc", "bal_acc", "sensitivity", "specificity", "auc", "auc_AD_HC", "auc_FTD_HC"]
C_TABLE = pd.DataFrame({n: {k: fmt(v, d, k) for k in KEYSD} for n, (v, d) in EVD.items()}).T
print("dementia (AD + FTD) vs HC, subject level, mean of repeats 1-3 [95 % CI]; benchmark 80.7 % (Stefanou 2025)\n")
print(C_TABLE[["acc", "sensitivity", "specificity", "auc"]].to_string())
_c = []
for k in ("acc", "auc"):
    t = paired(EVD["dementia soft vote"][1], EVD["3-class soft vote by-product"][1], k)
    _c.append(dict(metric=k, diff=EVD["dementia soft vote"][0][k] - EVD["3-class soft vote by-product"][0][k],
                   lo=t["lo"], hi=t["hi"], p=t["p"]))
C_TESTS = pd.DataFrame(_c)
print("\ndedicated soft vote minus the 3-class soft vote's by-product (paired bootstrap)\n")
print(C_TESTS.round(3).to_string(index=False))


dementia (AD + FTD) vs HC, subject level, mean of repeats 1-3 [95 % CI]; benchmark 80.7 % (Stefanou 2025)

                                               acc           sensitivity           specificity                   auc
dementia GAT                  0.814 [0.742, 0.883]  0.842 [0.757, 0.921]  0.759 [0.609, 0.897]  0.887 [0.814, 0.950]
band power                    0.807 [0.727, 0.875]  0.825 [0.729, 0.904]  0.770 [0.632, 0.897]  0.878 [0.799, 0.940]
dementia soft vote            0.826 [0.750, 0.898]  0.864 [0.780, 0.932]  0.747 [0.575, 0.897]  0.898 [0.827, 0.955]
3-class GAT by-product        0.788 [0.712, 0.856]  0.791 [0.706, 0.870]  0.782 [0.632, 0.908]  0.871 [0.791, 0.940]
3-class soft vote by-product  0.807 [0.731, 0.879]  0.819 [0.729, 0.898]  0.782 [0.632, 0.908]  0.889 [0.818, 0.950]

dedicated soft vote minus the 3-class soft vote's by-product (paired bootstrap)

metric  diff     lo    hi     p
   acc 0.019 -0.011 0.049 0.277
   auc 0.009 -0.006 0.025 0.273


In [5]:
# ===========================================================================
# CELL 5 — THE FINAL 3-CLASS MODEL: TWO STAGES
# ===========================================================================
_d = DEM["dementia soft vote"][0]                                       # (3, 88) P(dementia), stage 1
FINAL = np.stack([1 - _d, _d * Q, _d * (1 - Q)], -1)                  # (3, 88, 3)
_d3 = 1 - M3["soft vote, PSD graphs (today)"][..., HC]
REF2 = np.stack([1 - _d3, _d3 * Q, _d3 * (1 - Q)], -1)                 # the same stage 2 on the 3-class soft vote
EVF = {f"two-stage: dementia soft vote x {STAGE2} (final)": evaluate(three, FINAL),
       f"two-stage: 3-class soft vote x {STAGE2}": evaluate(three, REF2),
       "soft vote, PSD graphs (today)": EV3["soft vote, PSD graphs (today)"]}
F_TABLE = pd.DataFrame({n: {k: fmt(v, d, k) for k in KEYS3} for n, (v, d) in EVF.items()}).T
print("3-class, subject level, mean of repeats 1-3 [95 % CI]\n")
print(F_TABLE[["acc", "macro_f1", "macro_recall", "recall_HC", "recall_AD", "recall_FTD"]].to_string())
_fin, _today = list(EVF.values())[0], EVF["soft vote, PSD graphs (today)"]
F_TESTS = pd.DataFrame([dict(metric=k, diff=_fin[0][k] - _today[0][k],
                             **{x: v for x, v in paired(_fin[1], _today[1], k).items() if x != "diff"})
                        for k in ("macro_recall", "acc", "macro_f1")])
print("\nfinal minus today's soft vote (paired bootstrap)\n")
print(F_TESTS.round(3).to_string(index=False))


3-class, subject level, mean of repeats 1-3 [95 % CI]

                                                                                       acc              macro_f1          macro_recall             recall_HC             recall_AD            recall_FTD
two-stage: dementia soft vote x C3, ds006036 (Experiment 10) (final)  0.606 [0.515, 0.693]  0.589 [0.497, 0.673]  0.597 [0.505, 0.682]  0.782 [0.632, 0.908]  0.574 [0.426, 0.713]  0.435 [0.261, 0.609]
two-stage: 3-class soft vote x C3, ds006036 (Experiment 10)           0.621 [0.530, 0.708]  0.611 [0.516, 0.696]  0.614 [0.521, 0.700]  0.770 [0.621, 0.908]  0.593 [0.453, 0.731]  0.478 [0.304, 0.652]
soft vote, PSD graphs (today)                                         0.625 [0.538, 0.705]  0.592 [0.502, 0.680]  0.600 [0.512, 0.684]  0.782 [0.632, 0.908]  0.685 [0.556, 0.806]  0.333 [0.174, 0.493]

final minus today's soft vote (paired bootstrap)

      metric   diff     lo    hi     p
macro_recall -0.003 -0.094 0.085 0.901
         acc

In [6]:
# ===========================================================================
# CELL 6 — THE SCOREBOARD
# ===========================================================================
def verdict(bar, lo, hi):
    return "level" if lo <= bar <= hi else "ahead" if bar < lo else "behind"


_st = json.loads((WORK_DIR / "results_gat" / "statistics.json").read_text(encoding="utf-8"))["models"]["network"]
_e10s = json.loads((EXP_DIR / "c3_ds006036" / "summary.json").read_text(encoding="utf-8"))["primary"]
_dv, _dd = EVD["dementia soft vote"]
_fv, _fd = list(EVF.values())[0]
rows = [("AD vs HC", "AUC", _st["auc"]["value"], *_st["auc"]["ci95"], 0.912, "Wang et al. 2026 (LOSO)"),
        ("AD vs HC", "accuracy", _st["acc"]["value"], *_st["acc"]["ci95"], 0.877, "Wang et al. 2026 (LOSO)"),
        ("dementia vs HC", "accuracy", _dv["acc"], *ci(_dd, "acc"), 0.807, "Stefanou et al. 2025 (AHEPA review)"),
        ("AD vs FTD", "AUC", _e10s["auc"], *_e10s["ci"], 0.651, "Mlinaric et al. 2026 (LOSO, nested)")]
if _e11["primary"]["adopted_as_stage_2"]:
    rows.append(("AD vs FTD", "AUC, both recordings", _e11["primary"]["auc"], *_e11["primary"]["ci"], 0.651,
                 "Mlinaric et al. 2026 (LOSO, nested)"))
rows += [("3-class", "accuracy", _fv["acc"], *ci(_fd, "acc"), 0.70, "AHEPA review, average of subject-independent studies"),
         ("3-class", "accuracy", _fv["acc"], *ci(_fd, "acc"), 0.76, "AHEPA review, best quoted (Chen 2023, not checked)"),
         ("3-class", "macro-F1", _fv["macro_f1"], *ci(_fd, "macro_f1"), 0.595, "AHEPA review, average F1"),
         ("3-class", "macro recall", _fv["macro_recall"], *ci(_fd, "macro_recall"), 0.611, "strict GitHub benchmark")]
SCOREBOARD = pd.DataFrame(rows, columns=["task", "metric", "ours", "lo", "hi", "bar", "source"])
SCOREBOARD["verdict"] = [verdict(r.bar, r.lo, r.hi) for r in SCOREBOARD.itertuples()]
print(SCOREBOARD.round(3).to_string(index=False))
print("\nlevel = the published value lies inside our 95 % CI. The 3-class and AD-vs-FTD rows use both recordings "
      "of each patient (ds006036 for stage 2); the published results use the eyes-closed recording only.")


          task       metric  ours    lo    hi   bar                                               source verdict
      AD vs HC          AUC 0.909 0.842 0.965 0.912                              Wang et al. 2026 (LOSO)   level
      AD vs HC     accuracy 0.840 0.760 0.914 0.877                              Wang et al. 2026 (LOSO)   level
dementia vs HC     accuracy 0.826 0.750 0.898 0.807                  Stefanou et al. 2025 (AHEPA review)   level
     AD vs FTD          AUC 0.641 0.515 0.753 0.651                  Mlinaric et al. 2026 (LOSO, nested)   level
       3-class     accuracy 0.606 0.515 0.693 0.700 AHEPA review, average of subject-independent studies  behind
       3-class     accuracy 0.606 0.515 0.693 0.760   AHEPA review, best quoted (Chen 2023, not checked)  behind
       3-class     macro-F1 0.589 0.497 0.673 0.595                             AHEPA review, average F1   level
       3-class macro recall 0.597 0.505 0.682 0.611                              strict GitHub b

In [7]:
# ===========================================================================
# CELL 7 — SAVE
# ===========================================================================
B_TABLE.to_csv(OUT_DIR / "b_twins.csv"); B_TESTS.to_csv(OUT_DIR / "b_tests.csv", index=False)
C_TABLE.to_csv(OUT_DIR / "dementia.csv"); C_TESTS.to_csv(OUT_DIR / "dementia_tests.csv", index=False)
F_TABLE.to_csv(OUT_DIR / "final_three_class.csv"); F_TESTS.to_csv(OUT_DIR / "final_tests.csv", index=False)
SCOREBOARD.to_csv(OUT_DIR / "scoreboard.csv", index=False)
np.savez_compressed(OUT_DIR / "final_oof.npz", final=FINAL, y=Y, stage2=STAGE2)
(OUT_DIR / "summary.json").write_text(json.dumps(dict(
    stage2=STAGE2, scoreboard=SCOREBOARD.round(4).to_dict("records"),
    b_tests=B_TESTS.round(4).to_dict("records"), final_tests=F_TESTS.round(4).to_dict("records")),
    indent=1, default=float), encoding="utf-8")
print(f"saved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


saved to C:\Projects\Major Project\results_final: b_tests.csv, b_twins.csv, dementia.csv, dementia_tests.csv, final_oof.npz, final_tests.csv, final_three_class.csv, scoreboard.csv, summary.json
